<a href="https://colab.research.google.com/github/APenn215/MSBD-566-Predictive-Modeling/blob/main/AP_MSBD566_Lecture3_DataPrep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MSBD 566 — Feature Engineering & Data Preparation
## From the Decision Table to a Model-Ready Dataset

**Dataset:** Diabetes 130-US Hospitals (our semester-long dataset)
**CRISP-DM Stage 3:** *"How do we make the data ready for modeling?"*

Everything we do tonight was **decided in Data Understanding**:

| What we found | What we do tonight | Step |
|---|---|---|
| `'?'` encodes missing values | Replace with NaN | 1 |
| `weight` / `medical_specialty` / `payer_code` mostly missing | Drop **columns**, not rows | 1 |
| ~2,400 encounters end in death or hospice | Remove — readmission is undefined | 1 |
| `examide`, `citoglipton` are all 'No' | Drop — zero information | 1 |
| Age is *ordered*; race, admission type are *labels* | Ordinal vs one-hot encoding | 2 |
| `diag_1` has 715 distinct ICD-9 codes | Group into clinical categories | 3 |
| Prior visits strongly linked to readmission (your Q3!) | Engineer utilization features | 4 |
| 30% of rows are repeat patients | Split train/test **by patient** | 5 |
| Distance/gradient models are scale-sensitive | Standardize — fit on train only | 6 |

**Pipeline: (1) Clean → (2) Encode → (3) Tame cardinality → (4) Create features → (5) Split by patient → (6) Scale & save.**


---
# 0. Setup: Load the Raw Data Fresh

We start from the **raw file** — not from anything left in memory from the first half. A preparation pipeline should run *raw → model-ready* in one clean pass, top to bottom.


In [26]:
import os
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)

DATA_URL = "https://archive.ics.uci.edu/static/public/296/diabetes+130-us+hospitals+for+years+1999-2008.zip"
if not os.path.exists("diabetic_data.csv"):
    urllib.request.urlretrieve(DATA_URL, "diabetes_data.zip")
    with zipfile.ZipFile("diabetes_data.zip") as z:
        z.extractall(".")

data = pd.read_csv("diabetic_data.csv")
print("Raw shape:", data.shape)

Raw shape: (101766, 50)


---
# 1. Clean: Execute the Decision Table

Every line below implements a decision we **made and justified** in Data Understanding. Notice each one carries a comment citing the *reason* — a cleaning script without reasons is a liability.


In [27]:
# --- 1a. Standardize the missing marker (finding: '?' means missing) ---
data = data.replace('?', np.nan)

# --- 1b. Drop columns that are mostly missing (97% / 49% / 40%) ---
# The COLUMNS are the problem, not the rows — dropping rows would discard half the data.
data = data.drop(columns=['weight', 'medical_specialty', 'payer_code'])

# --- 1c. Drop zero-variance columns (every single value is 'No' — no information) ---
data = data.drop(columns=['examide', 'citoglipton'])

# --- 1d. Remove encounters ending in death or hospice (readmission is UNDEFINED) ---
# discharge_disposition_id codes: 11 = Expired; 13, 14, 19, 20, 21 = hospice / expired variants
died_or_hospice_ids = [11, 13, 14, 19, 20, 21]
n_before = len(data)
data = data[~data['discharge_disposition_id'].isin(died_or_hospice_ids)].copy()
print(f"Removed {n_before - len(data)} death/hospice encounters")

# --- 1e. Remove the 3 'Unknown/Invalid' gender rows (the invisible-bar discovery) ---
data = data[data['gender'] != 'Unknown/Invalid'].copy()

# --- 1f. Re-create our binary target (the business decision: 30-day window) ---
data['readmit_30'] = (data['readmitted'] == '<30').astype(int)

print("Shape after cleaning:", data.shape)
print("30-day readmission rate:", round(data['readmit_30'].mean() * 100, 1), "%")

Removed 2423 death/hospice encounters
Shape after cleaning: (99340, 46)
30-day readmission rate: 11.4 %


---
# 2. Encode: Turning Categories into Numbers

The one question that decides *how* to convert each text column: **does it have a natural order?**

- **Ordinal** (ordered) → map to ordered integers — e.g., `age`
- **Nominal** (just labels) → **one-hot encoding**: one 0/1 column per category — e.g., `race`


In [28]:
# --- 2a. ORDINAL: age brackets -> ordered integers 0..9 ---
age_order = ['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)',
             '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)']
age_map = {bracket: i for i, bracket in enumerate(age_order)}
data['age_ord'] = data['age'].map(age_map)

data[['age', 'age_ord']].head()

,age,age_ord
0,[0-10),0
1,[10-20),1
2,[20-30),2
3,[30-40),3
4,[40-50),4


In [29]:
# --- 2b. NOMINAL: one-hot encode unordered categoricals ---
# race has ~2% missing -> keep 'Missing' as its own category (missingness can be informative)
data['race'] = data['race'].fillna('Missing')

n_cols_before = data.shape[1]
data = pd.get_dummies(
    data,
    columns=['race', 'gender', 'admission_type_id'],
    prefix=['race', 'gender', 'admtype'],
    dtype=int,
)
print(f"Columns before: {n_cols_before}  ->  after one-hot: {data.shape[1]}")

# peek at the new columns
[c for c in data.columns if c.startswith(('race_', 'gender_', 'admtype_'))]

Columns before: 47  ->  after one-hot: 60


['race_AfricanAmerican',
 'race_Asian',
 'race_Caucasian',
 'race_Hispanic',
 'race_Missing',
 'race_Other',
 'gender_Female',
 'gender_Male',
 'admtype_1',
 'admtype_2',
 'admtype_3',
 'admtype_4',
 'admtype_5',
 'admtype_6',
 'admtype_7',
 'admtype_8']

---
# 3. Tame Cardinality: 715 Diagnosis Codes → 9 Clinical Groups

One-hot encoding `diag_1` naively would create **715 sparse, noisy columns** — an open invitation to overfitting. The fix comes from **medicine, not math**: ICD-9 codes come in ranges, and we group them into 9 clinical categories — the standard grouping published *with this dataset* (Strack et al., 2014):

| ICD-9 range | Group |
|---|---|
| 390–459, 785 | Circulatory |
| 460–519, 786 | Respiratory |
| 520–579, 787 | Digestive |
| 250.xx | Diabetes |
| 800–999 | Injury |
| 710–739 | Musculoskeletal |
| 580–629, 788 | Genitourinary |
| 140–239 | Neoplasms |
| everything else (incl. V/E codes) | Other |


In [30]:
# First, SEE the problem — your number from the activity:
print("Distinct ICD-9 codes in diag_1:", data['diag_1'].nunique())

Distinct ICD-9 codes in diag_1: 715


In [31]:
def icd9_group(code):
    """Map a raw ICD-9 code string to one of 9 clinical groups (Strack et al., 2014)."""
    if pd.isna(code):
        return 'Other'
    if str(code).startswith(('V', 'E')):          # supplementary / external-cause codes
        return 'Other'
    value = float(code)
    if 390 <= value <= 459 or value == 785:  return 'Circulatory'
    if 460 <= value <= 519 or value == 786:  return 'Respiratory'
    if 520 <= value <= 579 or value == 787:  return 'Digestive'
    if int(value) == 250:                    return 'Diabetes'
    if 800 <= value <= 999:                  return 'Injury'
    if 710 <= value <= 739:                  return 'Musculoskeletal'
    if 580 <= value <= 629 or value == 788:  return 'Genitourinary'
    if 140 <= value <= 239:                  return 'Neoplasms'
    return 'Other'

data['diag_1_group'] = data['diag_1'].apply(icd9_group)
print(data['diag_1_group'].value_counts())

# one-hot the 9 groups (this is now CHEAP), and drop the raw high-cardinality columns
data = pd.get_dummies(data, columns=['diag_1_group'], prefix='diag1', dtype=int)
data = data.drop(columns=['diag_1', 'diag_2', 'diag_3'])

diag_1_group
Circulatory        29680
Other              17813
Respiratory        13934
Digestive           9333
Diabetes            8661
Injury              6851
Genitourinary       5002
Musculoskeletal     4935
Neoplasms           3131
Name: count, dtype: int64


---
# 4. Create Features: Encoding Hypotheses

A good feature **encodes a hypothesis** — state it in clinical language first, *then* write the arithmetic.

| Hypothesis | Feature |
|---|---|
| "Past utilization predicts future utilization" — *your Q3 finding: 1.22 vs 0.56* | `total_prior_visits` |
| "An unstable medication regimen suggests an unstable patient" | `num_med_changes` |
| "Actively managed diabetes differs from incidental" | `on_diabetes_med`, `regimen_changed` |
| "Tame the zero-inflated skew from the histograms" | `log1p(...)` transforms |


In [32]:
# --- 4a. Total prior utilization (your Q3 discovery, now a feature) ---
data['total_prior_visits'] = (data['number_outpatient']
                              + data['number_emergency']
                              + data['number_inpatient'])

# --- 4b. Medication instability: how many drugs were dosed Up or Down this encounter? ---
med_cols = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
            'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',
            'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide',
            'insulin', 'glyburide-metformin', 'glipizide-metformin',
            'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']
data['num_med_changes'] = data[med_cols].isin(['Up', 'Down']).sum(axis=1)

# --- 4c. Simple clinical flags ---
data['on_diabetes_med'] = (data['diabetesMed'] == 'Yes').astype(int)
data['regimen_changed'] = (data['change'] == 'Ch').astype(int)

# --- 4d. Log-transform the zero-inflated counts (histograms, Week 2) ---
for col in ['number_outpatient', 'number_emergency', 'number_inpatient', 'total_prior_visits']:
    data[f'log_{col}'] = np.log1p(data[col])

# The habit: sanity-check every new feature against the target immediately.
data.groupby('readmit_30')['total_prior_visits'].mean().round(2)

,total_prior_visits
readmit_30,
0,1.09
1,2.02


---
# 5. Split by PATIENT — the Promise Comes Due

30% of rows are repeat patients. A random **row** split would put the same person in both train and test — the model would partly *recognize patients* instead of learning medicine (**data leakage**).

**Fix:** `GroupShuffleSplit` with `patient_nbr` as the group — every patient lives entirely on one side.
We split **before** scaling, because anything *fitted* must be fitted on training data only.


In [33]:
pip install scikit-learn

In [34]:
from sklearn.model_selection import GroupShuffleSplit

# Assemble features X and target y.
# Drop: IDs (never features!), raw text columns already encoded, and the target itself.
leftover_text = data.select_dtypes(include='object').columns.tolist()
X = data.drop(columns=['encounter_id', 'patient_nbr', 'readmitted', 'readmit_30',
                       'age', 'admission_source_id', 'discharge_disposition_id'] + leftover_text)
y = data['readmit_30']
groups = data['patient_nbr']

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# VERIFY the promise: zero patient overlap between train and test
overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Patients appearing in BOTH train and test: {len(overlap)}")
assert len(overlap) == 0, "Leakage! The same patient is on both sides."

print(f"Readmit rate — train: {y_train.mean():.3f} | test: {y_test.mean():.3f}")

Train: (79567, 42) | Test: (19773, 42)
Patients appearing in BOTH train and test: 0
Readmit rate — train: 0.114 | test: 0.113


---
# 6. Scale (Fit on Train Only) & Save Everything

Distance- and gradient-based models (kNN, SVM, regularized regression, neural nets) need features on comparable scales; trees don't. So we save **both** versions.

**The rule:** the scaler is **fitted on training data only** — fitting on all data leaks test information into training.


In [35]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train),   # FIT on train...
                              columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test),         # ...only TRANSFORM test
                             columns=X_test.columns, index=X_test.index)

print("Scaled. Example — num_lab_procedures now has train mean ~0, std ~1:")
print(X_train_scaled['num_lab_procedures'].describe().round(2).loc[['mean', 'std']])

Scaled. Example — num_lab_procedures now has train mean ~0, std ~1:
mean    0.0
std     1.0
Name: num_lab_procedures, dtype: float64


---
# 🏆 Challenge Questions (homework)

> ⚠️ **No AI for these.** Do not paste these into ChatGPT, Claude, Copilot, or any other AI tool. Struggle with them yourself first — the struggle IS the learning, and these are designed to be solvable with only what we did tonight plus the pandas documentation. If you're truly stuck after a real attempt, bring your attempt (not a blank cell) to office hours or the discussion board.

**C1 — Prove there's no leakage.** Without using the `overlap` variable from Step 5, write your **own** check that no `patient_nbr` appears in both train and test. *(Hint: `np.intersect1d`, or set operations on `groups.iloc[...]`.)*

**C2 — Interrogate a feature.** Compute the 30-day readmission rate for encounters with `num_med_changes == 0` versus `>= 1`. Does the "unstable regimen" hypothesis survive contact with the data? Add one markdown sentence interpreting the result.

**C3 — Engineer YOUR feature.** Create one new feature we did not build tonight. In a markdown cell, state the clinical hypothesis first (*why* should it predict?), then the code, then a groupby sanity-check against `readmit_30` — the exact habit from Step 4.

**C4 — Count the damage avoided.** How many columns would `pd.get_dummies` have created if we had one-hot encoded the raw `diag_1`, `diag_2`, and `diag_3` instead of grouping? Show the computation. *(You'll need to re-load the raw CSV in a scratch cell.)*

**C5 (stretch) — The debatable column.** `insulin` takes values No / Steady / Up / Down. Make the case for treating it as **ordinal** AND the case for **one-hot**. Pick one, implement it, and defend your choice in two sentences. There is no single right answer — there is only a **documented decision**.


In [36]:
# Save the model-ready dataset — every future week loads THESE files.
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
X_train_scaled.to_csv('X_train_scaled.csv', index=False)
X_test_scaled.to_csv('X_test_scaled.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print("Saved 6 files. Final feature matrix:", X_train.shape[1], "columns")
X_train.head()

Saved 6 files. Final feature matrix: 42 columns


,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses,age_ord,race_AfricanAmerican,race_Asian,race_Caucasian,race_Hispanic,race_Missing,race_Other,gender_Female,gender_Male,admtype_1,admtype_2,admtype_3,admtype_4,admtype_5,admtype_6,admtype_7,admtype_8,diag1_Circulatory,diag1_Diabetes,diag1_Digestive,diag1_Genitourinary,diag1_Injury,diag1_Musculoskeletal,diag1_Neoplasms,diag1_Other,diag1_Respiratory,total_prior_visits,num_med_changes,on_diabetes_med,regimen_changed,log_number_outpatient,log_number_emergency,log_number_inpatient,log_total_prior_visits
0,1,41,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0.000000,0.0,0.000000,0.000000
1,3,59,0,18,0,0,0,9,1,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,1,1,0.000000,0.0,0.000000,0.000000
2,2,11,5,13,2,0,1,6,2,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,3,0,1,0,1.098612,0.0,0.693147,1.386294
3,2,44,1,16,0,0,0,7,3,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,1,1,0.000000,0.0,0.000000,0.000000
4,1,51,0,8,0,0,0,5,4,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,1,0.000000,0.0,0.000000,0.000000


In [37]:
# C1 — your code here

train_patients = groups.iloc[train_idx]
test_patients = groups.iloc[test_idx]

Shared_patients = np.intersect1d(train_patients, test_patients)

print("Shared patients:",len(Shared_patients))

Shared patients: 0


In [38]:
data["num_med_changes"]>=1

#separate

no_changes = data[data['num_med_changes']==0]
with_changes = data[data['num_med_changes']>=1]

# cal rates using mean

no_changes_rate = no_changes['readmit_30'].mean()
with_changes_rate = with_changes['readmit_30'].mean()

#display

print("No medication changes:", no_changes_rate *100, "%")
print("One or more medication changes:", with_changes_rate * 100, "%")

No medication changes: 10.654037610619469 %
One or more medication changes: 13.35677276091784 %


Patients with one or more medication changes had a higher 30-day readmission rate 13.36% compared to patients with no medication changes 10.65%, supporting the hypothesis that changes in medication regimens may be associated with increased readmission risk.

**C3 hypothesis**

I believe patients with longer hospital stays may have a higher risk of being readmitted within 30 days because they may have more serious health conditions or require more complex treatment

In [39]:
# C3 — your hypothesis (markdown), code, and sanity-check here

data["long_stay"] = (data["time_in_hospital"] > 3).astype(int)
data.groupby("long_stay")["readmit_30"].mean() * 100

,readmit_30
long_stay,
0,9.859799
1,12.827425


Patients who stayed in the hospital longer than 3 days had a higher 30-day readmission rate (12.83%) compared to patients who stayed 3 days or less (9.86%). This supports my hypothesis that longer hospital stays may be associated with an increased risk of readmission.


In [40]:
# C4 — your code here
raw_data = pd.read_csv("diabetic_data.csv")

diagnosis_cols = raw_data[["diag_1", "diag_2", "diag_3"]]

encoded_diagnosis = pd.get_dummies(diagnosis_cols)

print("Total columns created:", encoded_diagnosis.shape[1])

Total columns created: 2256


One-hot encoding the three raw diagnosis columns would have created 2,256 columns. Grouping the diagnosis codes into broader clinical categories helped reduce the number of features and made the dataset easier to manage

In [41]:
# C5 (stretch) — your implementation and justification here
# One-hot encode the insulin column

insulin_encoded = pd.get_dummies(data["insulin"], prefix="insulin")


insulin_encoded.head()

,insulin_Down,insulin_No,insulin_Steady,insulin_Up
0,False,True,False,False
1,False,False,False,True
2,False,True,False,False
3,False,False,False,True
4,False,False,True,False


Ordinal encoding would keep the insulin information in one column, but it could create a numerical order that does not reflect the actual treatment categories. I chose one-hot encoding because it allows the model to recognize each insulin category separately without assuming one treatment change is greater than another

---
# 📌 Before Next Class

1. **Complete BOTH notebooks** — the Week 2 EDA notebook and this one, including the challenge questions (your own work — no AI).
2. **Push both to your GitHub repository** (the notebooks — not the generated CSVs).
3. **Next week:** Advanced Regression Modeling — the dataset you just built meets its first models.
